# Hybrid Digital–Analog GPT-2 Pipeline — Phase Selector

This notebook uses **only code committed to**:

`https://github.com/williamhwangweiju/projection-sensitivity-mapping`

It does not upload or overlay an implementation ZIP and does not patch source
files. The setup fetches the selected branch, resets the Colab checkout to
that commit, and invokes the phase functions directly from the repository.

Pipeline order: **Phase 0 (HWA fine-tune) → Phase 1 (sensitivity) →
proxy scores → Phase 1.5 (digital selection) → Phase 2 (fidelity trace) →
Phase 3 (placement) → validation → Phase 4 (quality + LAMBADA) →
energy analysis**.

Set `USE_HWA = False` for the vanilla post-training-quantization (PTQ)
contrast — the same pipeline on pretrained weights, matching the original
seed_42 results.


In [ ]:
#@title 1. Experiment and phase settings
from pathlib import Path

REPO_URL = "https://github.com/williamhwangweiju/projection-sensitivity-mapping.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

RUN_MODE = "full"  #@param ["smoke", "full"]
SEED = 42  #@param {type:"integer"}

# Hardware-aware deployment: Phase 0 fine-tunes under the deployment noise
# model and every later phase loads that checkpoint. Set False for the PTQ
# contrast (pretrained weights, no Phase 0).
USE_HWA = True  #@param {type:"boolean"}

# Set a phase to 1 to run it, or 0 to skip it. Enabled phases always execute
# in pipeline order: Phase 0 -> 1 -> 2 -> 3 -> 4. Phase 1 automatically
# includes proxy scoring and digital operating-point selection (Phase 1.5);
# Phase 4 automatically includes the energy analysis.
RUN_PHASE0 = 1  #@param {type:"integer"}
RUN_PHASE1 = 1  #@param {type:"integer"}
RUN_PHASE2 = 1  #@param {type:"integer"}
RUN_PHASE3 = 1  #@param {type:"integer"}
RUN_PHASE4 = 1  #@param {type:"integer"}

MOUNT_GOOGLE_DRIVE = True  #@param {type:"boolean"}
PREFER_GPU = True  #@param {type:"boolean"}
RUN_TESTS = False  #@param {type:"boolean"}

PHASE_FLAGS = {
    "phase0": RUN_PHASE0 if USE_HWA else 0,
    "phase1": RUN_PHASE1,
    "phase2": RUN_PHASE2,
    "phase3": RUN_PHASE3,
    "phase4": RUN_PHASE4,
}
invalid_phase_flags = {
    name: value
    for name, value in PHASE_FLAGS.items()
    if value not in (0, 1)
}
if invalid_phase_flags:
    raise ValueError(
        "Every phase flag must be exactly 0 or 1: "
        f"{invalid_phase_flags}"
    )

PHASES_TO_RUN = [name for name, enabled in PHASE_FLAGS.items() if enabled == 1]
if not PHASES_TO_RUN:
    raise ValueError("Enable at least one phase in Cell 1.")

# Leave blank to use the newest matching artifact in Drive.
PHASE1_ARTIFACT = ""  #@param {type:"string"}
PROXY_ARTIFACT = ""  #@param {type:"string"}
OPERATING_POINTS_ARTIFACT = ""  #@param {type:"string"}
PHASE2_TRACE_ARTIFACT = ""  #@param {type:"string"}
PHASE3_MANIFEST_ARTIFACT = ""  #@param {type:"string"}

PROJECT_DIR = Path("/content/projection-sensitivity-mapping")
IBM_3DSIM_DIR = PROJECT_DIR / "simulators" / "ibm_3d_cim"
IBM_3DSIM_URL = "https://github.com/IBM/3D-CiM-LLM-Inference-Simulator.git"

DRIVE_ROOT = Path("/content/drive/MyDrive/projection-sensitivity-mapping-hybrid-auto")
LOCAL_ROOT = Path("/content/projection-sensitivity-mapping-hybrid-auto")


In [ ]:
#@title 2. Mount Drive and clone/reset the GitHub repository
import os
import shutil
import subprocess

def run_checked(command, *, cwd=None, env=None):
    command = [str(value) for value in command]
    print("+", " ".join(command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    STORAGE_ROOT = DRIVE_ROOT
else:
    STORAGE_ROOT = LOCAL_ROOT

MODEL_VARIANT = "hwa" if USE_HWA else "ptq"
RESULTS_ROOT = STORAGE_ROOT / "results" / RUN_MODE / MODEL_VARIANT / f"seed_{SEED}"
LOG_ROOT = STORAGE_ROOT / "logs"
CACHE_ROOT = STORAGE_ROOT / "cache"
CONFIG_ROOT = STORAGE_ROOT / "configs"

for path in (STORAGE_ROOT, RESULTS_ROOT, LOG_ROOT, CACHE_ROOT, CONFIG_ROOT):
    path.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(CACHE_ROOT / "huggingface")
os.environ["HF_DATASETS_CACHE"] = str(CACHE_ROOT / "huggingface" / "datasets")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if (PROJECT_DIR / ".git").is_dir():
    run_checked(["git", "fetch", "origin", BRANCH], cwd=PROJECT_DIR)
    run_checked(["git", "checkout", "-B", BRANCH, f"origin/{BRANCH}"], cwd=PROJECT_DIR)
    run_checked(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=PROJECT_DIR)
    run_checked(["git", "clean", "-fd"], cwd=PROJECT_DIR)
else:
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    run_checked([
        "git", "clone", "--branch", BRANCH, "--depth", "1",
        "--recurse-submodules", "--shallow-submodules",
        REPO_URL, PROJECT_DIR,
    ])

run_checked(["git", "submodule", "sync", "--recursive"], cwd=PROJECT_DIR)
run_checked(["git", "submodule", "update", "--init", "--recursive"], cwd=PROJECT_DIR)

if not (IBM_3DSIM_DIR / "setup.py").is_file():
    if IBM_3DSIM_DIR.exists():
        shutil.rmtree(IBM_3DSIM_DIR)
    IBM_3DSIM_DIR.parent.mkdir(parents=True, exist_ok=True)
    run_checked(["git", "clone", "--depth", "1", IBM_3DSIM_URL, IBM_3DSIM_DIR])

COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR, text=True
).strip()
STATUS = subprocess.check_output(
    ["git", "status", "--short"], cwd=PROJECT_DIR, text=True
).strip()

print("Repository:", PROJECT_DIR)
print("Branch:", BRANCH)
print("Commit:", COMMIT)
print("Working tree clean:", STATUS == "")
print("Results root:", RESULTS_ROOT)

In [ ]:
#@title 3. Install dependencies, AIHWKit, and IBM 3D-CIM
import platform
import subprocess
import sys
from pathlib import Path


def shell(command: str) -> None:
    print("+", command)
    subprocess.run(["bash", "-lc", command], check=True)


shell("apt-get update -qq")
shell(
    "DEBIAN_FRONTEND=noninteractive apt-get install -y -qq "
    "libopenblas-dev build-essential cmake ninja-build graphviz"
)
shell(f"{sys.executable} -m pip install -q --upgrade pip setuptools wheel")

python_dependencies = [
    "transformers>=4.30,<5",
    "datasets>=2.14,<4",
    "pandas>=2.2,<3",
    "PyYAML>=6",
    "matplotlib>=3.7",
    "tqdm>=4.65",
    "pytest>=7",
    "pydot>=1.4",
]
shell(
    f"{sys.executable} -m pip install -q --upgrade --no-cache-dir "
    + " ".join(f"'{dependency}'" for dependency in python_dependencies)
)

py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
has_gpu_runtime = subprocess.run(
    ["bash", "-lc", "command -v nvidia-smi >/dev/null 2>&1"],
    check=False,
).returncode == 0

if PREFER_GPU and has_gpu_runtime and py_tag in {"cp310", "cp311", "cp312"}:
    wheel_name = (
        f"aihwkit-1.1.0-{py_tag}-{py_tag}-"
        "manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl"
    )
    wheel_url = (
        "https://aihwkit-gpu-demo.s3.us-east.cloud-object-storage.appdomain.cloud/"
        + wheel_name
    )
    wheel_path = Path("/content") / wheel_name
    shell(f"wget -q --show-progress -O '{wheel_path}' '{wheel_url}'")
    shell(
        f"{sys.executable} -m pip install -q "
        f"--force-reinstall --no-deps '{wheel_path}'"
    )
    AIHWKIT_INSTALL_KIND = "official GPU wheel"
else:
    shell(
        f"{sys.executable} -m pip install -q "
        "--upgrade --no-cache-dir 'aihwkit==1.1.0'"
    )
    AIHWKIT_INSTALL_KIND = "PyPI CPU package"

shell(
    f"{sys.executable} -m pip install -q "
    f"--no-deps -e '{IBM_3DSIM_DIR}'"
)

print("Python:", platform.python_version())
print("AIHWKit installation:", AIHWKIT_INSTALL_KIND)
print("IBM 3D-CIM editable package:", IBM_3DSIM_DIR)

In [ ]:
#@title 4. Verify AIHWKit, threedsim, and CUDA
import importlib
import importlib.metadata
import os
import subprocess
import sys

import torch
import transformers
import datasets
import aihwkit
from aihwkit.nn import AnalogLinear

IBM_3DSIM_SRC = IBM_3DSIM_DIR / "src"
simulator_src = str(IBM_3DSIM_SRC)

if simulator_src not in sys.path:
    sys.path.insert(0, simulator_src)

pythonpath_parts = [str(PROJECT_DIR), simulator_src]
existing_pythonpath = os.environ.get("PYTHONPATH", "")
if existing_pythonpath:
    pythonpath_parts.append(existing_pythonpath)
os.environ["PYTHONPATH"] = os.pathsep.join(pythonpath_parts)

importlib.invalidate_caches()
import threedsim

RUNTIME_DEVICE = "cuda" if PREFER_GPU and torch.cuda.is_available() else "cpu"

layer = AnalogLinear(2, 2)
try:
    if RUNTIME_DEVICE == "cuda":
        layer = layer.cuda()
        probe_input = torch.tensor([[0.1, 0.2]], device="cuda")
    else:
        probe_input = torch.tensor([[0.1, 0.2]])
    probe_output = layer(probe_input)
    if RUNTIME_DEVICE == "cuda":
        torch.cuda.synchronize()
except Exception as exc:
    print("AIHWKit GPU probe failed; falling back to CPU:", repr(exc))
    RUNTIME_DEVICE = "cpu"
    layer = AnalogLinear(2, 2)
    probe_output = layer(torch.tensor([[0.1, 0.2]]))

subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import threedsim; "
            "from threedsim.accelerator import Accelerator, AcceleratorConfig; "
            "from threedsim.mapping import Mapper, MapStrategy, Strategy; "
            "print('Subprocess threedsim:', threedsim.__file__)"
        ),
    ],
    env=os.environ.copy(),
    check=True,
)

print({
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "cuda_build": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "selected_device": RUNTIME_DEVICE,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "aihwkit": getattr(
        aihwkit, "__version__", importlib.metadata.version("aihwkit")
    ),
    "threedsim": threedsim.__file__,
})
print("AIHWKit probe:", probe_output.detach().cpu().numpy())

In [ ]:
#@title 5. Edit and create the complete Drive-backed experiment config
import yaml

full_config = PROJECT_DIR / "configs/full_pipeline/gpt2_hybrid_3dcim.yaml"
smoke_config = PROJECT_DIR / "configs/full_pipeline/gpt2_hybrid_3dcim_smoke.yaml"

BASE_CONFIG = full_config if RUN_MODE == "full" else smoke_config
if not BASE_CONFIG.is_file():
    raise FileNotFoundError(
        "The selected hybrid config is not present in the GitHub checkout: "
        f"{BASE_CONFIG}"
    )

# The repository configuration is the source of truth. Put ONLY the values
# you want to change in this block; it is deep-merged over the repository
# config, so every repository key you do not override (hwa_training, proxy
# scoring, static_fisher policies, lambada, cost_model, ...) stays active.
# NOTE: lists replace rather than merge — overriding e.g. phase4.policies
# here replaces the full repository list.
COLAB_CONFIG_OVERRIDES_YAML = r"""
{}
# --- examples ---
# hwa_training:
#   max_steps: 2000
#   precision: bf16          # use fp32 on a T4
#   dataset:
#     batch_size: 8          # lower this if the GPU runs out of memory
# profiling:
#   num_seeds: 5
#   proxy:
#     max_batches: 32
# phase4:
#   max_operating_points: 3
#   lambada:
#     max_examples: 1000
# cost_model:
#   e_adc_pj: 1.5
"""


def deep_merge(base, overrides):
    """Recursively merge Colab overrides without discarding new base keys."""
    for key, value in overrides.items():
        if (
            key in base
            and isinstance(base[key], dict)
            and isinstance(value, dict)
        ):
            deep_merge(base[key], value)
        else:
            base[key] = value
    return base


config = yaml.safe_load(BASE_CONFIG.read_text(encoding="utf-8"))
overrides = yaml.safe_load(COLAB_CONFIG_OVERRIDES_YAML)
if not isinstance(config, dict):
    raise ValueError("Repository config must contain a YAML mapping.")
if overrides is None:
    overrides = {}
if not isinstance(overrides, dict):
    raise ValueError("COLAB_CONFIG_OVERRIDES_YAML must contain a YAML mapping.")
config = deep_merge(config, overrides)

config.setdefault("experiment", {})
config["experiment"]["seed"] = int(SEED)
config["experiment"]["placement_seed"] = int(SEED)
config.setdefault("model", {})
config["model"]["device"] = RUNTIME_DEVICE

phase_output_roots = {
    "hwa_training": RESULTS_ROOT / "phase0",
    "phase1": RESULTS_ROOT / "phase1",
    "digital_selection": RESULTS_ROOT / "phase1_5_digital_selection",
    "phase2": RESULTS_ROOT / "phase2",
    "phase3": RESULTS_ROOT / "phase3",
    "phase4": RESULTS_ROOT / "phase4",
}
for section, output_root in phase_output_roots.items():
    if section in config and isinstance(config[section], dict):
        config[section]["output_root"] = str(output_root)

# Keep the Phase 0 contract consistent: model.checkpoint must equal
# <hwa_training.output_root>/checkpoint_final when HWA is used, and must be
# null for the PTQ contrast.
HWA_CHECKPOINT_DIR = RESULTS_ROOT / "phase0" / "checkpoint_final"
if USE_HWA:
    config.setdefault("hwa_training", {})["enabled"] = True
    config["model"]["checkpoint"] = str(HWA_CHECKPOINT_DIR)
else:
    if "hwa_training" in config:
        config["hwa_training"]["enabled"] = False
    config["model"]["checkpoint"] = None

COLAB_CONFIG = CONFIG_ROOT / (
    f"gpt2_hybrid_{RUN_MODE}_{'hwa' if USE_HWA else 'ptq'}_seed{SEED}.yaml"
)
COLAB_CONFIG.write_text(
    yaml.safe_dump(config, sort_keys=False),
    encoding="utf-8",
)

print("Repository base config:", BASE_CONFIG)
print("Generated Drive config:", COLAB_CONFIG)
print("Selected phases:", PHASES_TO_RUN)
print("HWA mode:", USE_HWA, "| checkpoint:", config["model"]["checkpoint"])
print("Phase 4 policies:", config.get("phase4", {}).get("policies"))
print(
    "Selection methods:",
    config.get("digital_selection", {}).get("methods"),
)


In [ ]:
#@title 6. Optional repository tests
import os
import subprocess
import sys

if RUN_TESTS:
    subprocess.run(
        [sys.executable, "-m", "pytest", "-q"],
        cwd=PROJECT_DIR,
        env=os.environ.copy(),
        check=True,
    )
    smoke_script = PROJECT_DIR / "scripts/smoke_aihwkit_contract.py"
    if smoke_script.is_file():
        subprocess.run(
            [sys.executable, str(smoke_script), "--device", RUNTIME_DEVICE],
            cwd=PROJECT_DIR,
            env=os.environ.copy(),
            check=True,
        )
    print("Repository tests passed.")
else:
    print("Tests skipped.")

In [ ]:
#@title 7. Resolve existing artifacts from Google Drive
from pathlib import Path

def newest(patterns, *, label, required=False):
    matches = []
    for pattern in patterns:
        matches.extend(RESULTS_ROOT.glob(pattern))
    matches = [path for path in matches if path.is_file()]
    matches.sort(key=lambda path: path.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(
            f"No {label} artifact found under {RESULTS_ROOT}. "
            f"Patterns: {patterns}"
        )
    return None

def explicit_or_latest(raw_path, patterns, label, required=False):
    if str(raw_path).strip():
        path = Path(str(raw_path).strip())
        if not path.is_file():
            raise FileNotFoundError(f"{label} does not exist: {path}")
        return path
    return newest(patterns, label=label, required=required)

PHASE1_PATH = explicit_or_latest(
    PHASE1_ARTIFACT,
    ["phase1/*sensitivity*.json", "phase1/**/*sensitivity*.json"],
    "Phase 1 profile",
)
# proxy_sensitivity files also match *sensitivity*; exclude them from Phase 1.
if PHASE1_PATH is not None and PHASE1_PATH.name.startswith("proxy_sensitivity"):
    candidates = sorted(
        (
            path
            for path in RESULTS_ROOT.glob("phase1/*sensitivity*.json")
            if not path.name.startswith("proxy_sensitivity")
        ),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    PHASE1_PATH = candidates[0] if candidates else None
PROXY_PATH = explicit_or_latest(
    PROXY_ARTIFACT,
    ["phase1/proxy_sensitivity_*.json"],
    "Proxy sensitivity sidecar",
)
OPERATING_POINTS_PATH = explicit_or_latest(
    OPERATING_POINTS_ARTIFACT,
    [
        "phase1_5_digital_selection/digital_operating_points.json",
        "phase1_5_digital_selection/**/digital_operating_points.json",
    ],
    "Phase 1.5 combined operating points",
)
TRACE_PATH = explicit_or_latest(
    PHASE2_TRACE_ARTIFACT,
    ["phase2/**/trace.npz"],
    "Phase 2 trace",
)
PHASE3_MANIFEST_PATH = explicit_or_latest(
    PHASE3_MANIFEST_ARTIFACT,
    ["phase3/phase3_manifest.json", "phase3/**/phase3_manifest.json"],
    "Phase 3 manifest",
)
print("Existing artifacts:")
print("Phase 1:", PHASE1_PATH)
print("Proxy sidecar:", PROXY_PATH)
print("Digital operating points:", OPERATING_POINTS_PATH)
print("Phase 2:", TRACE_PATH)
print("Phase 3:", PHASE3_MANIFEST_PATH)
if USE_HWA:
    print(
        "Phase 0 checkpoint present:",
        HWA_CHECKPOINT_DIR.is_dir(),
        "|",
        HWA_CHECKPOINT_DIR,
    )

# The executable multi-phase runner is defined immediately below.

RUNNER_SOURCE = r'''
import json
import os
import sys
from pathlib import Path

repo = Path(os.environ["PSM_PROJECT_DIR"])
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

actions = json.loads(os.environ["PSM_ACTIONS"])
config = Path(os.environ["PSM_CONFIG"])


def optional(name):
    value = os.environ.get(name, "").strip()
    return None if not value else Path(value)


phase1 = optional("PSM_PHASE1")
proxy = optional("PSM_PROXY")
points = optional("PSM_POINTS")
trace = optional("PSM_TRACE")
phase3 = optional("PSM_PHASE3")


def require(path, label):
    if path is None or not path.is_file():
        raise FileNotFoundError(
            f"{label} artifact is required but was not produced in this run "
            f"and was not found in Drive: {path}"
        )
    return path


def run_phase0():
    from experiments.phase0_hwa_training.run_hwa_training import main
    return main(config)


def run_phase1():
    from experiments.phase1_sensitivity.run_aihwkit_profiling import main
    from experiments.phase1_sensitivity.analyze_results import main as analyze
    result = main(config)
    analyze(result)
    return result


def run_proxy(phase1_path):
    from src.common.config import load_yaml

    cfg = load_yaml(config)
    if not bool(cfg.get("profiling", {}).get("proxy", {}).get("enabled", False)):
        print("Proxy scoring disabled in the configuration; skipping.")
        return None
    from experiments.phase1_sensitivity.run_proxy_sensitivity import main
    return main(config, phase1_path)


def run_digital_selection(phase1_path, proxy_path):
    from experiments.phase1_5_digital_selection.select_digital_operating_points import (
        main as select_base,
    )
    from experiments.phase1_5_digital_selection.select_greedy_marginal import (
        main as select_greedy,
    )
    from src.common.config import load_yaml, resolve_path

    base = select_base(config, phase1_path, proxy_path)
    cfg = load_yaml(config)
    output = (
        resolve_path(cfg["digital_selection"]["output_root"])
        / "greedy_marginal_points.json"
    )
    select_greedy(config, phase1_path, output, base)
    # Return the COMBINED artifact (base points + appended greedy points),
    # matching scripts/run_full_pipeline.py, so no selection method's
    # operating points are dropped downstream.
    return base


def run_phase2():
    from experiments.phase2_fidelity.run_fidelity_model import main
    return main(config)


def run_phase3(phase1_path, points_path, trace_path, proxy_path):
    from experiments.phase3_baselines.run_baseline_mappings import main
    return main(config, phase1_path, points_path, trace_path, proxy_path)


def validate(phase1_path, points_path, trace_path, phase3_path):
    from scripts.validate_pipeline_contracts import validate_pipeline
    validate_pipeline(config, phase1_path, points_path, trace_path, phase3_path)


def run_phase4(phase1_path, points_path, trace_path, phase3_path):
    from experiments.phase4_quality.run_hybrid_quality import main
    return main(config, phase1_path, points_path, trace_path, phase3_path)


def run_energy(points_path, phase3_path):
    from src.common.config import load_yaml, resolve_path

    cfg = load_yaml(config)
    if "cost_model" not in cfg:
        print("No cost_model section; skipping the energy analysis.")
        return None
    from experiments.phase4_quality.analyze_energy_quality import main

    frontier = (
        resolve_path(cfg["phase4"]["output_root"])
        / "quality_vs_budget_frontier.csv"
    )
    return main(config, points_path, phase3_path, frontier)


for action in actions:
    print(f"\n===== Running {action} =====")
    if action == "phase0":
        checkpoint = run_phase0()
        print("Phase 0 checkpoint:", checkpoint)
    elif action == "phase1":
        phase1 = run_phase1()
        proxy = run_proxy(phase1)
        points = run_digital_selection(phase1, proxy)
        print("Phase 1:", phase1)
        print("Proxy sidecar:", proxy)
        print("Digital operating points:", points)
    elif action == "phase2":
        trace = run_phase2()
        print("Phase 2:", trace)
    elif action == "phase3":
        phase3 = run_phase3(
            require(phase1, "Phase 1"),
            require(points, "Phase 1.5"),
            require(trace, "Phase 2"),
            proxy,
        )
        print("Phase 3:", phase3)
    elif action == "phase4":
        phase1 = require(phase1, "Phase 1")
        points = require(points, "Phase 1.5")
        trace = require(trace, "Phase 2")
        phase3 = require(phase3, "Phase 3")
        validate(phase1, points, trace, phase3)
        phase4 = run_phase4(phase1, points, trace, phase3)
        print("Phase 4:", phase4)
        energy = run_energy(points, phase3)
        print("Energy frontier:", energy)
    else:
        raise ValueError(f"Unsupported phase: {action}")

print("\nSelected phases complete:", actions)
'''


In [ ]:
#@title 8. Run the selected phases
from datetime import datetime
import json
import os
import subprocess
import sys

env = os.environ.copy()
env.update({
    "PYTHONUNBUFFERED": "1",
    "HF_HOME": os.environ["HF_HOME"],
    "HF_DATASETS_CACHE": os.environ["HF_DATASETS_CACHE"],
    "TOKENIZERS_PARALLELISM": "false",
    "PYTHONPATH": os.environ["PYTHONPATH"],
})

phase_label = "_".join(PHASES_TO_RUN)
log_path = LOG_ROOT / (
    f"{phase_label}_{RUN_MODE}_seed{SEED}_"
    f"{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
)

run_env = env.copy()
run_env.update({
    "PSM_PROJECT_DIR": str(PROJECT_DIR),
    "PSM_ACTIONS": json.dumps(PHASES_TO_RUN),
    "PSM_CONFIG": str(COLAB_CONFIG),
    "PSM_PHASE1": "" if PHASE1_PATH is None else str(PHASE1_PATH),
    "PSM_PROXY": "" if PROXY_PATH is None else str(PROXY_PATH),
    "PSM_POINTS": "" if OPERATING_POINTS_PATH is None else str(OPERATING_POINTS_PATH),
    "PSM_TRACE": "" if TRACE_PATH is None else str(TRACE_PATH),
    "PSM_PHASE3": "" if PHASE3_MANIFEST_PATH is None else str(PHASE3_MANIFEST_PATH),
})

command = [sys.executable, "-u", "-c", RUNNER_SOURCE]
print("Phases:", PHASES_TO_RUN)
print("Configuration:", COLAB_CONFIG)
print("Log:", log_path)

with log_path.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(
        command,
        cwd=PROJECT_DIR,
        env=run_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        log.write(line)
        log.flush()
    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"Selected phases failed with exit code {return_code}. "
        f"The complete traceback is above and in {log_path}."
    )

print("Completed phases:", PHASES_TO_RUN)
print("Results:", RESULTS_ROOT)
print("Log:", log_path)


In [ ]:
#@title 9. Inspect current artifacts and selected digital projections
import json
import pandas as pd
from IPython.display import display

def show_csv(path, columns=None):
    if not path.is_file():
        print("Not found:", path)
        return
    print(path)
    frame = pd.read_csv(path)
    if columns:
        selected = [column for column in columns if column in frame.columns]
        if selected:
            frame = frame[selected]
    display(frame)

phase0_metadata = RESULTS_ROOT / "phase0" / "hwa_metadata.json"
if phase0_metadata.is_file():
    payload = json.loads(phase0_metadata.read_text(encoding="utf-8"))
    print("Phase 0 final loss:", payload.get("final_loss"))
    history = payload.get("eval_history", [])
    if history:
        display(pd.DataFrame(history))
else:
    print("No Phase 0 metadata yet.")

greedy_json = (
    RESULTS_ROOT
    / "phase1_5_digital_selection"
    / "greedy_marginal_points.json"
)
greedy_csv = greedy_json.with_suffix(".csv")

if greedy_json.is_file():
    payload = json.loads(greedy_json.read_text(encoding="utf-8"))
    points = payload.get("operating_points", [])
    recommendation = payload.get("recommended_operating_point", {})
    print("Greedy operating points:", len(points))
    print(
        "Recommended digital set:",
        recommendation.get("digital_projection_ids"),
    )
    print(
        "Recommendation reason:",
        recommendation.get("recommendation_reason"),
    )
    show_csv(
        greedy_csv,
        [
            "budget_value",
            "promoted_projection_id",
            "digital_projection_count",
            "digital_mac_fraction",
            "digital_parameter_fraction",
            "measured_nominal_nll",
            "delta_nll_nominal_vs_digital",
            "marginal_nll_gain",
            "capacity_feasible",
            "digital_projection_ids",
        ],
    )
else:
    print("No greedy operating-point artifact yet.")

for name in [
    "proxy_rank_correlation.csv",
    "phase3_summary.csv",
    "nominal_hybrid_frontier.csv",
    "hybrid_quality_summary.csv",
    "paired_policy_summary.csv",
    "energy_quality_frontier.csv",
]:
    matches = sorted(
        RESULTS_ROOT.rglob(name),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if matches:
        print("\n", name)
        show_csv(matches[0])

figure = sorted(RESULTS_ROOT.rglob("energy_quality_pareto.png"))
if figure:
    from IPython.display import Image
    display(Image(filename=str(figure[-1])))


## Usage

1. In Cell 1, choose `USE_HWA` (hardware-aware checkpoint) or the PTQ
   contrast, and set each `RUN_PHASE...` value to `1` or `0`.
2. In Cell 5, put only the values you want to change in
   `COLAB_CONFIG_OVERRIDES_YAML`; the repository config supplies everything
   else (Phase 0 recipe, proxy scoring, `static_fisher`, LAMBADA,
   `cost_model`).
3. Run Cells 1–7 after starting a new runtime.
4. Run Cell 8 to execute every enabled phase in pipeline order.
5. Run Cell 9 to inspect outputs (Phase 0 eval history, greedy frontier,
   proxy rank correlations, quality summaries, energy Pareto figure).

Phase 0 resumes automatically from its newest step checkpoint if the session
was preempted; rerun Cell 8 with `RUN_PHASE0 = 1`. With `RUN_PHASE0 = 0` and
`USE_HWA = True`, the Phase 0 checkpoint must already exist under the
results directory from a previous session.

When a prerequisite phase is disabled, the notebook uses the newest matching
artifact already stored under the selected results directory. Set an explicit
artifact path in Cell 1 if you do not want the newest matching artifact.
HWA and PTQ artifacts live in separate trees:
`results/<mode>/hwa/seed_<seed>` versus `results/<mode>/ptq/seed_<seed>`, so
the two variants can never overwrite or cross-contaminate each other. Runs
recorded before this separation lived at `results/<mode>/seed_<seed>`; move
vanilla artifacts into the `ptq` subtree (and any Phase 0 output into the
`hwa` subtree) once, e.g.:

```
mv results/full/seed_42/phase0 results/full/hwa/seed_42/phase0   # HWA checkpoint
mv results/full/seed_42       results/full/ptq/seed_42           # vanilla artifacts
```
